In [21]:
import pandas as pd
import numpy as np
import os

# -------------------------------
# 1. Height Corruption Fixer
# -------------------------------
def fix_height(ht):
    if (pd.isna(ht)):
        return np.nan
    
    ht = str(ht).strip()

    if ("-" in ht):
        parts = ht.split("-")
        
        if (len(parts) == 2):
            left = parts[0]
            right = parts[1]

            # Case 1: Standard numeric format (e.g., "6-2" -> 6 feet 2 inches)
            if (left.isdigit() and right.isdigit()):
                return (int(left) * 12) + int(right)

            month_map = {
                "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
                "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
                "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
            }

            left_str = left[:3].title()
            right_str = right[:3].title()

            # Case 2: Excel corrupted Day-Month (e.g., "11-May" -> 5 feet 11 inches)
            if (left.isdigit() and right_str in month_map):
                inches = int(left)
                feet = month_map[right_str]
                return (feet * 12) + inches

            # Case 3: Excel corrupted Month-Day (e.g., "May-11" -> 5 feet 11 inches)
            if (left_str in month_map and right.isdigit()):
                feet = month_map[left_str]
                inches = int(right)
                return (feet * 12) + inches

    return np.nan

# -------------------------------
# 2. File Loading & Merge Setup
# -------------------------------
data_dir = '../data/historical_training_data/'

all_combine_data = []
all_stats_data = []

for year in range(2021, 2027):
    if (year == 2026):
        combine_file_path = '../data/2026Combine.csv'
    else:
        combine_file_path = f'{data_dir}{year}Combine.csv'
        
    if (os.path.exists(combine_file_path)):
        combine_df = pd.read_csv(combine_file_path)
        combine_df['Draft_Year'] = year
        all_combine_data.append(combine_df)
        
    stats_file_path = f'{data_dir}{year}Stats.csv'
    if (os.path.exists(stats_file_path)):
        stats_df = pd.read_csv(stats_file_path, header=1)
        stats_df['Draft_Year'] = year
        all_stats_data.append(stats_df)

master_combine = pd.concat(all_combine_data, ignore_index=True)
master_stats = pd.concat(all_stats_data, ignore_index=True)

# -------------------------------
# 3. Clean Names to Ensure Perfect Joins
# -------------------------------
master_combine['Player'] = master_combine['Player'].str.replace(r'[^a-zA-Z\s.-]', '', regex=True).str.strip()
master_stats['Player'] = master_stats['Player'].str.replace(r'[^a-zA-Z\s.-]', '', regex=True).str.strip()

# -------------------------------
# 4. The LEFT Join (Crucial for 2026 Class)
# -------------------------------
merged_df = pd.merge(
    master_combine, 
    master_stats, 
    on=['Player', 'Draft_Year'], 
    how='left',
    suffixes=('', '_stats')
)

feature_columns = [
    'Player', 'Pos', 'Draft_Year', 'Ht', 'Wt', '40yd', 
    'Vertical', 'Bench', 'Broad Jump', '3Cone', 'Shuttle'
]
target_columns = ['G', 'wAV']

# Initialize target columns with NaN if they drop out
for col in target_columns:
    if (col not in merged_df.columns):
        merged_df[col] = np.nan

all_columns = feature_columns + target_columns
clean_df = merged_df[all_columns].copy()

# -------------------------------
# 5. Apply Height Fix and Convert Numerics
# -------------------------------
clean_df['Ht_Inches'] = clean_df['Ht'].apply(fix_height)
clean_df = clean_df.drop(columns=['Ht'])

clean_df['Years_In_League'] = 2026 - clean_df['Draft_Year']

clean_df['wAV'] = pd.to_numeric(clean_df['wAV'], errors='coerce').fillna(0)
clean_df['G'] = pd.to_numeric(clean_df['G'], errors='coerce').fillna(0)

# -------------------------------
# 6. Calculate Average wAV
# -------------------------------
avg_wav_list = []
for index, row in clean_df.iterrows():
    years = row['Years_In_League']
    wav = row['wAV']
    
    if (years > 0):
        avg_wav_list.append(wav / years)
    else:
        avg_wav_list.append(np.nan)

clean_df['Avg_wAV_Per_Season'] = avg_wav_list

# -------------------------------
# 7. Drop Kickers and Punters Explicitly
# -------------------------------
kicker_punter_indices = []
for index, row in clean_df.iterrows():
    if (row["Pos"] == "K" or row["Pos"] == "P"):
        kicker_punter_indices.append(index)

clean_df = clean_df.drop(kicker_punter_indices).reset_index(drop=True)

# -------------------------------
# 8. Handle Missing Athletic Data
# -------------------------------
# Drop columns that are heavily missing (>50% missing data)
clean_df = clean_df.drop(columns=['Bench', '3Cone', 'Shuttle'])

# The highly predictive traits we want to save
traits_to_save = ['40yd', 'Vertical', 'Broad Jump', 'Wt']
unique_positions = clean_df['Pos'].unique()

for trait in traits_to_save:
    for pos in unique_positions:
        # 1. Isolate the players at this specific position
        pos_mask = (clean_df['Pos'] == pos)
        
        # 2. Calculate the average for this trait at this position
        pos_avg = clean_df.loc[pos_mask, trait].mean()
        
        # 3. Loop through and fill the blanks for this position
        for index, row in clean_df.iterrows():
            if (row['Pos'] == pos):
                if (pd.isna(row[trait])):
                    clean_df.at[index, trait] = pos_avg

# 4. Drop any stragglers (e.g., if a player is missing a trait and their entire position group also skipped it)
clean_df = clean_df.dropna().reset_index(drop=True)

# -------------------------------
# Final Model-Ready Diagnostics
# -------------------------------
print(f"Final Model-Ready Records: {len(clean_df)}")
print(f"Shape: {clean_df.shape}\n")
print("Remaining Missing Values:")
print(clean_df.isna().sum())

clean_df.head()

# -------------------------------
# Diagnostics
# -------------------------------
print(f"Successfully merged and cleaned {len(clean_df)} player records.")
print(f"Shape: {clean_df.shape}\n")
print("Missing values per column:")
print(clean_df.isna().sum())

clean_df.head()

Final Model-Ready Records: 1682
Shape: (1682, 12)

Remaining Missing Values:
Player                0
Pos                   0
Draft_Year            0
Wt                    0
40yd                  0
Vertical              0
Broad Jump            0
G                     0
wAV                   0
Ht_Inches             0
Years_In_League       0
Avg_wAV_Per_Season    0
dtype: int64
Successfully merged and cleaned 1682 player records.
Shape: (1682, 12)

Missing values per column:
Player                0
Pos                   0
Draft_Year            0
Wt                    0
40yd                  0
Vertical              0
Broad Jump            0
G                     0
wAV                   0
Ht_Inches             0
Years_In_League       0
Avg_wAV_Per_Season    0
dtype: int64


,Player,Pos,Draft_Year,Wt,40yd,Vertical,Broad Jump,G,wAV,Ht_Inches,Years_In_League,Avg_wAV_Per_Season
0,Jonathan Adams,WR,2021,210.0,4.59,39.0,132.0,0.0,0.0,74.0,5,0.0
1,Paulson Adebo,CB,2021,198.0,4.45,36.5,121.0,64.0,22.0,73.0,5,4.4
2,Giles Amos,TE,2021,250.0,5.14,28.5,111.0,0.0,0.0,75.0,5,0.0
3,Otis Anderson,RB,2021,179.0,4.63,36.0,113.0,0.0,0.0,67.0,5,0.0
4,Jack Anderson,OL,2021,314.0,5.28,29.5,105.0,15.0,2.0,76.0,5,0.4
